In [ ]:
# base.py

from abc import ABC, abstractmethod
from dataclasses import dataclass

@dataclass
class Item:
    sentence:  str
    condition: str

@dataclass
class Batch:
    items: list

class Phenomenon(ABC):
    @property
    @abstractmethod
    def name(self) -> str: ...

    @property
    @abstractmethod
    def condition_labels(self) -> list: ...

    @abstractmethod
    def generate_batches(self) -> list: ...

In [ ]:
# utils.py

MARKER = {
    ("NOM", True):  "이",
    ("NOM", False): "가",
    ("ACC", True):  "을",
    ("ACC", False): "를"
}

def has_batchim(noun: str) -> bool:
    code = ord(noun[-1])
    if not (0xAC00 <= code <= 0xD7A3):
        raise ValueError(f"Last character of '{noun}' is not a Hangul syllable")
    return (code - 0xAC00) % 28 != 0

def attach_marker(noun: str, case: str) -> str:
    return noun + MARKER[(case, has_batchim(noun))]

In [ ]:
# case_marking.py

from itertools import product as iproduct

DEFAULT_SUBJECTS = ["선생님", "아버지", "학생", "아이"]
DEFAULT_VERB_OBJECTS = {
    "먹었다": ["사과", "빵", "밥", "과자"],
    "읽었다": ["책", "신문", "소설", "잡지"],
    "샀다":   ["선물", "기념품", "커피", "꽃"],
    "찾았다": ["안경", "열쇠", "가방", "지갑"]
}
CASE_CONDITIONS = ["NOM-ACC", "ACC-NOM", "NOM-NOM", "ACC-ACC"]

def _make_sentence(subject, subj_case, obj, obj_case, verb):
    return f"{attach_marker(subject, subj_case)} {attach_marker(obj, obj_case)} {verb}"

class CaseMarkingPhenomenon(Phenomenon):
    def __init__(self, subjects = None, verb_objects = None):
        self.subjects     = subjects     or DEFAULT_SUBJECTS
        self.verb_objects = verb_objects or DEFAULT_VERB_OBJECTS

    @property
    def name(self): return "Case Marking (NOM/ACC)"

    @property
    def condition_labels(self): return CASE_CONDITIONS

    def generate_batches(self):
        case_map = {
            "NOM-ACC": ("NOM", "ACC"), "ACC-NOM": ("ACC", "NOM"),
            "NOM-NOM": ("NOM", "NOM"), "ACC-ACC": ("ACC", "ACC")
        }
        batches = []
        for verb, objs in self.verb_objects.items():
            for subj, obj in iproduct(self.subjects, objs):
                items = []
                for cond in CASE_CONDITIONS:
                    sc, oc = case_map[cond]
                    items.append(Item(
                        sentence  = _make_sentence(subj, sc, obj, oc, verb),
                        condition = cond,
                    ))
                batches.append(Batch(items = items))
        return batches

In [ ]:
# honorific.py

from itertools import product as iproduct

DEFAULT_HON_NOUNS   = ["선생님", "교수님", "선배님", "사장님", "아버지", "어머니", "할아버지", "할머니"]
DEFAULT_PLAIN_NOUNS = ["학생", "제자", "후배", "아이", "친구", "동생", "남동생", "여동생"]
DEFAULT_VERB_PAIRS  = [
    ("찾았다",   "찾으셨다"),
    ("불렀다",   "부르셨다"),
    ("기다렸다", "기다리셨다"),
    ("만났다",   "만나셨다"),
    ("칭찬했다", "칭찬하셨다"),
    ("도왔다",   "도우셨다")
]

HON_CONDITIONS = [
    "HON.SUBJ+HON.V",
    "HON.SUBJ+PLAIN.V",
    "PLAIN.SUBJ+HON.V",
    "PLAIN.SUBJ+PLAIN.V"
]

class HonorificPhenomenon(Phenomenon):
    def __init__(self, hon_nouns = None, plain_nouns = None, verb_pairs = None):
        self.hon_nouns   = hon_nouns   or DEFAULT_HON_NOUNS
        self.plain_nouns = plain_nouns or DEFAULT_PLAIN_NOUNS
        self.verb_pairs  = verb_pairs  or DEFAULT_VERB_PAIRS

    @property
    def name(self): return "Subject Honorific Agreement"

    @property
    def condition_labels(self): return HON_CONDITIONS

    def generate_batches(self):
        batches = []
        for hon, plain, (v_plain, v_hon) in iproduct(
            self.hon_nouns, self.plain_nouns, self.verb_pairs
        ):
            batches.append(Batch(items=[
                Item(f"{attach_marker(hon,   'NOM')} {attach_marker(plain, 'ACC')} {v_hon}",   "HON.SUBJ+HON.V"),
                Item(f"{attach_marker(hon,   'NOM')} {attach_marker(plain, 'ACC')} {v_plain}", "HON.SUBJ+PLAIN.V"),
                Item(f"{attach_marker(plain, 'NOM')} {attach_marker(hon,   'ACC')} {v_hon}",   "PLAIN.SUBJ+HON.V"),
                Item(f"{attach_marker(plain, 'NOM')} {attach_marker(hon,   'ACC')} {v_plain}", "PLAIN.SUBJ+PLAIN.V")
            ]))
        return batches

REGISTRY = {
    "case_marking": CaseMarkingPhenomenon,
    "honorific":    HonorificPhenomenon
}

In [ ]:
# SLOR.py

import torch
import pickle
from dataclasses import dataclass
from transformers import AutoModelForCausalLM, AutoTokenizer

UNIGRAM_MAP = {
    "EleutherAI/polyglot-ko-1.3b":    "polyglot-ko-1.3b",
    "EleutherAI/polyglot-ko-3.8b":    "polyglot-ko-1.3b",
    "EleutherAI/polyglot-ko-5.8b":    "polyglot-ko-1.3b",
    "LGAI-EXAONE/EXAONE-4.0-1.2B":   "EXAONE-4.0-1.2B",
    "kakaocorp/kanana-1.5-2.1b-base": "kanana-1.5-2.1b-base"
}

UNIGRAM_DIR = "./results/unigram"

@dataclass
class ScoredItem:
    item: Item
    slor: float
    supr: float

@dataclass
class ScoredBatch:
    scored_items: list

class SLORScorer:
    def __init__(self, model_name: str, device: str | None = None):
        if device is None:
            if torch.cuda.is_available():            device = "cuda"
            elif torch.backends.mps.is_available(): device = "mps"
            else:                                   device = "cpu"
        self.device = device
        print(f"Loading {model_name} on {device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        dtype = torch.float32 if device == "cpu" else torch.float16
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=dtype
        ).to(device)
        self.model.eval()
        self.eos_id = self.tokenizer.eos_token_id
        self._unigram_log_probs = self._load_unigram_log_probs(model_name)
        print("Ready.")

    def _load_unigram_log_probs(self, model_name: str) -> dict:
        if model_name not in UNIGRAM_MAP:
            raise ValueError(
                f"No unigram mapping for '{model_name}'.\n"
                f"Available: {list(UNIGRAM_MAP.keys())}"
            )
        tag      = UNIGRAM_MAP[model_name]
        pkl_path = f"{UNIGRAM_DIR}/unigram_{tag}_wiki.pkl"
        print(f"Loading unigram from {pkl_path}...")
        with open(pkl_path, "rb") as f:
            payload = pickle.load(f)
        log_pu = payload["log_pu"]
        print(f"Unigram loaded. vocab_size={len(log_pu)}, "
              f"total_tokens={payload['meta']['total_tokens']:,}")
        return log_pu

    @torch.no_grad()
    def _sentence_log_prob(self, sentence: str):
        token_ids = self.tokenizer.encode(sentence, add_special_tokens = False)
        input_ids = torch.tensor(
            [[self.eos_id] + token_ids], device=self.device
        )
        logits    = self.model(input_ids).logits
        log_probs = torch.log_softmax(logits[0], dim = -1).cpu()
        total     = sum(log_probs[i, tid].item() for i, tid in enumerate(token_ids))
        return total, token_ids

    def _slor(self, sentence: str) -> float:
        log_prob, token_ids = self._sentence_log_prob(sentence)
        length = len(token_ids)
        unigram_sum = sum(
            self._unigram_log_probs.get(tid, -20.0) for tid in token_ids
        )
        return (log_prob - unigram_sum) / length

    def _mean_surprisal(self, sentence: str) -> float:
        log_prob, token_ids = self._sentence_log_prob(sentence)
        return -log_prob / len(token_ids)

    def score_batches(self, batches):
        results = []
        for i, batch in enumerate(batches):
            if (i + 1) % 25 == 0 or i == 0:
                print(f"  Scoring batch {i + 1}/{len(batches)}...")
            scored_items = [
                ScoredItem(
                    item = item,
                    slor = self._slor(item.sentence),
                    supr = self._mean_surprisal(item.sentence),
                )
                for item in batch.items
            ]
            results.append(ScoredBatch(scored_items = scored_items))
        return results

In [ ]:
# plot.py

import matplotlib.pyplot as plt
import numpy as np

def plot_violins(
    scored_batches,
    condition_labels,
    title         = "",
    save_path     = None,
    ylabel        = "SLOR",
    metric        = "slor",
    show_baseline = False):

    data = {label: [] for label in condition_labels}
    for sb in scored_batches:
        for si in sb.scored_items:
            value = si.slor if metric == "slor" else si.supr
            data[si.item.condition].append(value)

    fig, ax = plt.subplots(figsize=(8, 5))
    positions   = list(range(1, len(condition_labels) + 1))
    violin_data = [data[label] for label in condition_labels]
    parts = ax.violinplot(violin_data, positions = positions, showmeans = True, showmedians = True)

    for pos, values in zip(positions, violin_data):
        jitter = np.random.default_rng(42).uniform(-0.05, 0.05, size=len(values))
        ax.scatter(pos + jitter, values, alpha = 0.3, s = 8, color = "black", zorder = 3)

    if show_baseline:
        ax.axhline(y = 1.0, color = "gray", linestyle = "--", linewidth = 0.8, label = "uniform baseline")
        ax.legend()

    ax.set_xticks(positions)
    ax.set_xticklabels(condition_labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title or "SLOR by Condition")
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi = 150, bbox_inches = "tight")
        print(f"Saved plot to {save_path}")
    else:
        plt.show()

In [ ]:
# stats.py

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, ttest_rel, shapiro

METRIC_META = {
    "slor": "SLOR",
    "supr": "Mean Surprisal"
}

HYPOTHESIS_PAIRS = {
    "honorific": [
        # (1) Match vs Mismatch
        ("HON.SUBJ+HON.V",    "HON.SUBJ+PLAIN.V"),
        ("HON.SUBJ+HON.V",    "PLAIN.SUBJ+HON.V"),
        ("PLAIN.SUBJ+PLAIN.V","HON.SUBJ+PLAIN.V"),
        ("PLAIN.SUBJ+PLAIN.V","PLAIN.SUBJ+HON.V"),
        # (2) Within Mismatch
        ("HON.SUBJ+PLAIN.V",  "PLAIN.SUBJ+HON.V")
    ],
    "case_marking": [
        # (1) Grammatical vs. Ungrammatical
        ("NOM-ACC", "ACC-NOM"),
        ("NOM-ACC", "NOM-NOM"),
        ("NOM-ACC", "ACC-ACC"),
        # (2) Within Ungrammatical
        ("ACC-NOM", "NOM-NOM"),
        ("ACC-NOM", "ACC-ACC"),
        ("NOM-NOM", "ACC-ACC")
    ]
}

def scored_batches_to_df(scored_batches):
    records = []
    for batch_idx, sb in enumerate(scored_batches):
        for si in sb.scored_items:
            records.append({
                "batch_idx": batch_idx,
                "condition": si.item.condition,
                "slor":      si.slor,
                "supr":      si.supr
            })
    return pd.DataFrame(records)

def condition_descriptives(df, metric):
    desc = (
        df.groupby("condition")[metric]
          .agg(n = "count", mean = "mean", median = "median", sd = "std")
          .reset_index()
          .round(4)
    )
    cond_order = list(
        df.groupby("condition")[metric].mean().sort_values(ascending = False).index
    )
    desc["condition"] = pd.Categorical(desc["condition"], categories=cond_order, ordered = True)
    return desc.sort_values("condition").reset_index(drop = True)

def run_shapiro(df, metric):
    results = {}
    for cond, group in df.groupby("condition"):
        stat, p = shapiro(group[metric].dropna())
        results[cond] = {"W": round(stat, 4), "p": round(p, 4), "normal": p > 0.05}
    return results

def run_friedman(df, metric):
    wide = (
        df.pivot_table(index = "batch_idx", columns = "condition", values = metric)
          .dropna()
    )
    conditions = list(wide.columns)
    stat, p = friedmanchisquare(*[wide[c].values for c in conditions])
    return {"statistic": stat, "p_value": p, "n_batches": len(wide), "conditions": conditions}

def _paired_ttest(x, y):
    t, p = ttest_rel(x, y)
    diff = x - y
    d    = diff.mean() / diff.std(ddof = 1)
    return t, p, abs(d)

def _sig_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def _effect_label_d(d):
    if d >= 0.8: return "large"
    if d >= 0.5: return "medium"
    if d >= 0.2: return "small"
    return "negligible"

def run_pairwise_ttest(df, metric, pairs):
    wide = (
        df.pivot_table(index = "batch_idx", columns = "condition", values = metric)
          .dropna()
    )
    rows = []
    for ca, cb in pairs:
        t, p_raw, d = _paired_ttest(wide[ca].values, wide[cb].values)
        rows.append({"cond_a": ca, "cond_b": cb, "t": t, "p_raw": p_raw, "cohen_d": d})
    result = pd.DataFrame(rows)
    k = len(result)
    result["p_bonf"]    = (result["p_raw"] * k).clip(upper=1.0)
    result["reject_05"] = result["p_bonf"] < 0.05
    result["sig"]       = result["p_bonf"].apply(_sig_stars)
    result["interp_d"]  = result["cohen_d"].apply(_effect_label_d)
    return result

def _report_one_metric(df, metric, phenomenon):
    label = METRIC_META[metric]
    print(f"\n  ▶ Metric: {label}\n")

    desc = condition_descriptives(df, metric)
    print(desc.to_string(index = False))

    print(f"\n  [Shapiro-Wilk Normality Test]")
    shapiro_results = run_shapiro(df, metric)
    for cond, res in shapiro_results.items():
        flag = "✓ normal" if res["normal"] else "✗ non-normal"
        print(f"  {cond:30s}  W={res['W']:.4f},  p={res['p']:.4f}  {flag}")

    fr = run_friedman(df, metric)
    print(f"\n  [Friedman Test]")
    print(f"  χ²({len(fr['conditions'])-1}) = {fr['statistic']:.4f},  "
          f"p = {fr['p_value']:.4e},  N_batches = {fr['n_batches']}")
    print("  → " + (
        "At least one condition differs significantly."
        if fr["p_value"] < 0.05 else
        "No significant difference among conditions."
    ))

    pairs = HYPOTHESIS_PAIRS[phenomenon]
    pw    = run_pairwise_ttest(df, metric, pairs)
    k     = len(pw)
    print(f"\n  [Paired t-test + Bonferroni]  "
          f"comparisons = {k},  α_bonf = {0.05/k:.4f}\n")

    pw_display = pw[["cond_a", "cond_b", "t", "p_raw", "p_bonf", "sig", "cohen_d", "interp_d"]].copy()
    pw_display["p_raw"]   = pw_display["p_raw"].map(lambda x: f"{x:.4e}")
    pw_display["p_bonf"]  = pw_display["p_bonf"].map(lambda x: f"{x:.4e}")
    pw_display["t"]       = pw_display["t"].map(lambda x: f"{x:.4f}")
    pw_display["cohen_d"] = pw_display["cohen_d"].map(lambda x: f"{x:.3f}")
    print(pw_display.to_string(index = False))

    sig_pairs = pw[pw["reject_05"]]
    print(f"\n  [Significant after Bonferroni  (p_bonf < .05)]")
    if sig_pairs.empty:
        print("  None.")
    else:
        for _, row in sig_pairs.iterrows():
            print(f"  {row['cond_a']:30s} vs  {row['cond_b']:30s}"
                  f"  {row['sig']:3s}  d={row['cohen_d']:.3f} ({row['interp_d']})")

    return {"descriptives": desc, "shapiro": shapiro_results, "friedman": fr, "pairwise": pw}

def run_full_report(scored_batches, phenomenon, model_name = ""):
    df  = scored_batches_to_df(scored_batches)
    sep = "=" * 64
    print(f"\n{sep}")
    print(f"  Statistical Report | {phenomenon.upper()} | {model_name}")
    print(sep)

    results = {}
    for metric in ("slor", "supr"):
        print(f"\n{'-' * 64}")
        results[metric] = _report_one_metric(df, metric, phenomenon)

    print(f"\n{sep}\n")
    return {"df": df, **results}

In [ ]:
# Models to Use

# EXAONE
# LGAI-EXAONE/EXAONE-4.0-1.2B

# Kanana
# kakaocorp/kanana-1.5-2.1b-base

# Polyglot-Ko
# EleutherAI/polyglot-ko-1.3b
# EleutherAI/polyglot-ko-3.8b
# EleutherAI/polyglot-ko-5.8b

In [ ]:
# Settings

MODEL_NAME = "EleutherAI/polyglot-ko-5.8b"
PHENOMENON = "honorific"   # "case_marking" or "honorific"

_tag  = MODEL_NAME.split("/")[-1]

SAVE_PATH_SLOR = f"./results/{PHENOMENON}/SLOR/{PHENOMENON}_SLOR_{_tag}.png"
SAVE_PATH_SUPR = f"./results/{PHENOMENON}/Surprisal/{PHENOMENON}_Surprisal_{_tag}.png"


In [ ]:
# Check Dataset

phenomenon = REGISTRY[PHENOMENON]()
batches    = phenomenon.generate_batches()
print(f"Total batches: {len(batches)},  Total sentences: {len(batches) * len(batches[0].items)}\n")

for i, batch in enumerate(batches[:3]):
    print(f"Batch {i}:")
    for item in batch.items:
        print(f"  [{item.condition:20s}] {item.sentence}")
    print()

In [ ]:
# Run Experiment

import os
import pandas as pd

phenomenon = REGISTRY[PHENOMENON]()
batches    = phenomenon.generate_batches()
print(f"Generated {len(batches)} batches ({len(batches) * len(batches[0].items)} sentences)")

scorer = SLORScorer(model_name = MODEL_NAME)

print("\nScoring...")
scored_batches = scorer.score_batches(batches)

_csv_dir  = f"{_base}/{PHENOMENON}/data"
_csv_path = f"{_csv_dir}/{PHENOMENON}_{_tag}.csv"
os.makedirs(_csv_dir, exist_ok = True)

_rows = []
for _batch_idx, _sb in enumerate(scored_batches):
    for _si in _sb.scored_items:
        _rows.append({
            "model":      MODEL_NAME,
            "phenomenon": PHENOMENON,
            "batch_idx":  _batch_idx,
            "condition":  _si.item.condition,
            "sentence":   _si.item.sentence,
            "slor":       _si.slor,
            "supr":       _si.supr
        })

df_raw = pd.DataFrame(_rows)
df_raw.to_csv(_csv_path, index = False, encoding = "utf-8-sig")
print(f"Saved CSV → {_csv_path}")
print(f"Shape: {df_raw.shape}  ({df_raw['condition'].nunique()} conditions × {len(batches)} batches)")

In [ ]:
# Run Stats & Plot

import os

os.makedirs(f"{_base}/{PHENOMENON}/SLOR",      exist_ok=True)
os.makedirs(f"{_base}/{PHENOMENON}/Surprisal", exist_ok=True)

results = run_full_report(
    scored_batches = scored_batches,
    phenomenon     = PHENOMENON,
    model_name     = MODEL_NAME
)

# Plot: SLOR
plot_violins(
    scored_batches,
    condition_labels = phenomenon.condition_labels,
    title     = f"{phenomenon.name} — {MODEL_NAME} (SLOR)",
    save_path = SAVE_PATH_SLOR,
    ylabel    = "SLOR",
    metric    = "slor"
)

# Plot: Mean Surprisal
plot_violins(
    scored_batches,
    condition_labels = phenomenon.condition_labels,
    title     = f"{phenomenon.name} — {MODEL_NAME} (Mean Surprisal)",
    save_path = SAVE_PATH_SUPR,
    ylabel    = "Mean Surprisal",
    metric    = "supr"
)